<a href="https://colab.research.google.com/github/tijanigambari2004-oss/TEEJAY-S-flower-classifier/blob/main/image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Load the flowers dataset (5 classes)
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = keras.utils.get_file("flower_photos", url, untar=True)

img_size = (224, 224)
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=img_size, batch_size=32)
val_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=img_size, batch_size=32)

class_names = train_ds.class_names
print(class_names)

# 2. Build the model: pretrained base + our own classifier head
base = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base.trainable = False  # freeze pretrained layers

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

inputs = keras.Input(shape=(224, 224, 3))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names))(x)
model = keras.Model(inputs, outputs)

# 3. Train
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"])

history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 4. Evaluate and save
model.evaluate(val_ds)
model.save("flower_classifier.keras")

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.
['flower_photos']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 86ms/step - accuracy: 0.9275 - loss: 0.0000e+00 - val_accuracy: 0.9632 - val_loss: 0.0000e+00
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 70ms/step - accuracy: 0.9200 - loss: 0.0000e+00 - val_accuracy: 0.9632 - val_loss: 0.0000e+00
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 70ms/step - accuracy: 0.9169 - loss: 0.0000e+00 - val_accuracy: 0.9632 - val_loss: 0.0000e+00
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 70ms/step - accuracy: 0.9240 - loss: 0.0000e+00 - val_accuracy: 0.9632 - val_loss: 0.0000e+00
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 71ms/step - accuracy: 0.9183 - loss: 0.0000e+00 - val_accuracy: 0.9632 - val_loss: 0.0000e+00
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.9632 - los

In [ ]:
import os
print(os.listdir(data_dir))

['flower_photos']


In [ ]:
import pathlib

data_dir = pathlib.Path(data_dir)
if (data_dir / "flower_photos").exists():
    data_dir = data_dir / "flower_photos"

print(os.listdir(data_dir))

['dandelion', 'LICENSE.txt', 'tulips', 'sunflowers', 'daisy', 'roses']


In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(224, 224), batch_size=32)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(224, 224), batch_size=32)

class_names = train_ds.class_names
print(class_names)

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [ ]:
import os
import pathlib
import tensorflow as tf
from tensorflow import keras

print(data_dir)
print(os.listdir(data_dir))

/root/.keras/datasets/flower_photos/flower_photos
['dandelion', 'LICENSE.txt', 'tulips', 'sunflowers', 'daisy', 'roses']


In [ ]:
data_dir = pathlib.Path(data_dir)
if (data_dir / "flower_photos").exists():
    data_dir = data_dir / "flower_photos"

print(os.listdir(data_dir))

['dandelion', 'LICENSE.txt', 'tulips', 'sunflowers', 'daisy', 'roses']


In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(224, 224), batch_size=32)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(224, 224), batch_size=32)

class_names = train_ds.class_names
print(class_names)

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Safety check: stop if the data is still wrong
assert len(class_names) == 5, f"Expected 5 classes, got {len(class_names)}: {class_names}"

# 1. Build the model
base = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base.trainable = False  # keep the pretrained knowledge fixed

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

inputs = keras.Input(shape=(224, 224, 3))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names))(x)   # 5 outputs, one per flower
model = keras.Model(inputs, outputs)

# 2. Compile
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"])

# 3. Train
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 4. Check accuracy on unseen images
loss, accuracy = model.evaluate(val_ds)
print(f"\nValidation accuracy: {accuracy:.2%}")

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 13s 84ms/step - accuracy: 0.6870 - loss: 0.8616 - val_accuracy: 0.8351 - val_loss: 0.4852
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 71ms/step - accuracy: 0.8420 - loss: 0.4471 - val_accuracy: 0.8692 - val_loss: 0.3996
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 71ms/step - accuracy: 0.8573 - loss: 0.3937 - val_accuracy: 0.8529 - val_loss: 0.4364
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 73ms/step - accuracy: 0.8801 - loss: 0.3344 - val_accuracy: 0.8774 - val_loss: 0.3643
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 72ms/step - accuracy: 0.8856 - loss: 0.3176 - val_accuracy: 0.8719 - val_loss: 0.3650
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.8719 - loss: 0.3650

Validation accuracy: 87.19%


In [ ]:
from google.colab import files
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# 1. Upload your pictures (a file picker will appear)
uploaded = files.upload()

# 2. Predict each one
for filename in uploaded.keys():
    img = Image.open(filename)
    img = ImageOps.exif_transpose(img)          # fixes sideways phone photos
    img = img.convert("RGB").resize((224, 224))

    img_array = np.expand_dims(np.array(img), axis=0)
    pred = model.predict(img_array, verbose=0)
    probs = tf.nn.softmax(pred[0]).numpy()

    top = np.argmax(probs)
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{class_names[top]} ({probs[top]:.0%})")
    plt.show()

    print(filename)
    for name, p in zip(class_names, probs):
        print(f"  {name}: {p:.1%}")
    print()

In [ ]:
import os
import numpy as np
from PIL import Image
from tensorflow import keras

print(os.listdir(data_dir))   # should show the 5 flower folders

(x_train, y_train), _ = keras.datasets.cifar10.load_data()

out = data_dir / "not_flower"
out.mkdir(exist_ok=True)

rng = np.random.default_rng(42)
idx = rng.choice(len(x_train), 700, replace=False)

for n, i in enumerate(idx):
    Image.fromarray(x_train[i]).resize((224, 224), Image.BICUBIC).save(out / f"cifar_{n}.jpg")

print("not_flower images:", len(os.listdir(out)))

In [ ]:
from google.colab import files
from PIL import ImageOps

uploaded = files.upload()
for k, filename in enumerate(uploaded.keys()):
    img = Image.open(filename)
    img = ImageOps.exif_transpose(img).convert("RGB").resize((224, 224))
    img.save(out / f"mine_{k}.jpg")

print("not_flower images now:", len(os.listdir(out)))

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(224, 224), batch_size=32)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(224, 224), batch_size=32)

class_names = train_ds.class_names
print(class_names)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

assert len(class_names) == 6, f"Expected 6 classes, got {len(class_names)}: {class_names}"

base = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base.trainable = False

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

inputs = keras.Input(shape=(224, 224, 3))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names))(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"])

history = model.fit(train_ds, validation_data=val_ds, epochs=5)

loss, accuracy = model.evaluate(val_ds)
print(f"\nValidation accuracy: {accuracy:.2%}")